# 02. Compass north calibrations summary

This notebook processes compass/north calibration annotation files exported from the waggle dance annotator.

For each compass calibration video, the annotated compass-north vectors are converted into video-frame angles. These angles are summarized with circular statistics and corrected for local magnetic declination to estimate the position of true geographic north in the video frame.

Outputs from this notebook are used by later notebooks to calibrate waggle run angles from horizontal and dome recordings.

This notebook does:

1. Load compass calibration CSV files.
2. Extract compass-north annotation vectors.
3. Convert vectors to video-frame angles.
4. Calculate the circular mean of magnetic north.
5. Correct magnetic north to true geographic north using magnetic declination.
6. Save a raw annotation-level file and a summary calibration file.
7. Run QC checks.

This notebook does not:

- Extract waggle runs from dance videos.
- Assign direct calibrations to dances.
- Process landmark-transfer calibrations.
- Combine horizontal and dome dance datasets.
- Perform biological/statistical analysis.

## Imports, paths, and magnetic declination

This cell defines the folder containing the compass calibration CSV files, the output file names, and the magnetic declination used to convert magnetic north to true geographic north.

The declination is set as a negative value because the local magnetic declination is west.

### Expected input folder

`HORIZONTAL_COMPASS_ROOT` should contain the compass calibration CSV files directly inside the folder.

The notebook uses files matching:

`*_waggle_annotations.csv`

Landmark annotation files are excluded later if their filename contains `landmark`.

The use of `glob()` rather than a recursive folder search is intentional here, because copied calibration files in subfolders can otherwise create duplicate calibration IDs.

In [2]:
from pathlib import Path
import ast
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
# Check that compass folder exists and its path
HORIZONTAL_COMPASS_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_compass_calibrations"
)

OUTPUT_CALIBRATION_SUMMARY_FILE = HORIZONTAL_COMPASS_ROOT / "calibration_summary_ROBUST.csv"
OUTPUT_CALIBRATION_RAW_FILE = HORIZONTAL_COMPASS_ROOT / "calibration_raw_annotations_ROBUST.csv"

DECLINATION_DEG = -21.27

print("Compass folder exists:", HORIZONTAL_COMPASS_ROOT.exists())
print("Compass folder:", HORIZONTAL_COMPASS_ROOT)

Compass folder exists: True
Compass folder: C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_compass_calibrations


In [10]:
def parse_annotation_list(value, column_name=""):
    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    value = re.sub(
        r"np\.float(?:16|32|64)?\(([^)]+)\)",
        r"\1",
        value
    )

    value = re.sub(r"\bnan\b", "None", value, flags=re.IGNORECASE)

    # Repair common missing opening bracket issue:
    # "(1, 2), (3, 4)]" -> "[(1, 2), (3, 4)]"
    if value.startswith("(") and value.endswith("]"):
        value = "[" + value

    try:
        return list(ast.literal_eval(value))
    except (ValueError, SyntaxError) as error:
        raise ValueError(
            f"Could not parse column '{column_name}': {value}"
        ) from error


def vector_to_video_angle_deg(u, v):
    """
    Video convention:
    0°   = top/up
    90°  = right
    180° = bottom/down
    270° = left
    """
    return np.degrees(np.arctan2(u, -v)) % 360


def circular_mean_deg(angles_deg):
    angles_rad = np.radians(angles_deg)

    mean_rad = np.arctan2(
        np.mean(np.sin(angles_rad)),
        np.mean(np.cos(angles_rad))
    )

    return np.degrees(mean_rad) % 360


def circular_resultant_length_deg(angles_deg):
    angles_rad = np.radians(angles_deg)

    return np.sqrt(
        np.mean(np.cos(angles_rad))**2
        + np.mean(np.sin(angles_rad))**2
    )


def circular_sd_deg(angles_deg):
    R = circular_resultant_length_deg(angles_deg)
    R = np.clip(R, 1e-12, 1)

    return np.degrees(np.sqrt(-2 * np.log(R)))


def clean_calibration_id(filename):
    name = Path(filename).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name

In [11]:
def extract_compass_angles_from_csv(csv_file):
    """
    Extract compass north angles from a calibration CSV.

    Important:
    This function intentionally ignores waggle_start_positions.
    For compass calibration, only waggle_directions are needed.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    for session_index, row in df.iterrows():

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        # Frames are useful but not required
        try:
            frames = parse_annotation_list(
                row["waggle_start_frames"],
                "waggle_start_frames"
            )
        except Exception:
            frames = []

        for i, direction in enumerate(directions, start=1):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            # Skip invalid zero vectors
            if np.hypot(u, v) < 1e-8:
                continue

            angle_deg = vector_to_video_angle_deg(u, v)

            frame = frames[i - 1] if i - 1 < len(frames) else np.nan

            records.append({
                "calibration_file": csv_file.name,
                "calibration_id": clean_calibration_id(csv_file.name),
                "annotation_number": len(records) + 1,
                "original_annotation_index": i,
                "annotation_session": session_index + 1,
                "frame": frame,
                "direction_u": u,
                "direction_v": v,
                "calibration_video_angle_deg": angle_deg
            })

    return pd.DataFrame(records)

### Process all compass calibration files

In [12]:
calibration_files = sorted(
    HORIZONTAL_COMPASS_ROOT.glob("*_waggle_annotations.csv")
)

# Optional: avoid accidentally processing landmark files if any are present
calibration_files = [
    file for file in calibration_files
    if "landmark" not in file.name.lower()
]

print(f"Found {len(calibration_files)} calibration files.")

raw_dfs = []
summary_rows = []
failed_files = []

for file in calibration_files:
    try:
        raw_df = extract_compass_angles_from_csv(file)

        if len(raw_df) == 0:
            raise ValueError("No valid compass direction vectors found.")

        raw_dfs.append(raw_df)

        angles = raw_df["calibration_video_angle_deg"].values

        magnetic_north_deg = circular_mean_deg(angles)
        true_north_video_deg = (magnetic_north_deg - DECLINATION_DEG) % 360

        n_annotations = len(raw_df)

        if n_annotations == 1:
            quality_flag = "only_one_annotation"
        elif n_annotations < 5:
            quality_flag = "few_annotations"
        else:
            quality_flag = "ok"

        summary_rows.append({
            "calibration_file": file.name,
            "calibration_id": clean_calibration_id(file.name),
            "calibration_video_name": raw_df["calibration_file"].iloc[0],
            "calibration_n_annotations": n_annotations,
            "calibration_mean_north_deg": magnetic_north_deg,
            "calibration_circular_sd_deg": circular_sd_deg(angles),
            "calibration_R": circular_resultant_length_deg(angles),
            "declination_deg": DECLINATION_DEG,
            "true_north_video_deg": true_north_video_deg,
            "calibration_quality_flag": quality_flag,
            "calibration_notes": ""
        })

    except Exception as error:
        failed_files.append({
            "calibration_file": file.name,
            "path": str(file),
            "error": str(error)
        })

calibration_raw_df = pd.concat(raw_dfs, ignore_index=True)
calibration_summary_df = pd.DataFrame(summary_rows)

calibration_raw_df.to_csv(OUTPUT_CALIBRATION_RAW_FILE, index=False)
calibration_summary_df.to_csv(OUTPUT_CALIBRATION_SUMMARY_FILE, index=False)

print(f"\nSaved robust raw calibration annotations to:\n{OUTPUT_CALIBRATION_RAW_FILE}")
print(f"\nSaved robust calibration summary to:\n{OUTPUT_CALIBRATION_SUMMARY_FILE}")

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    display(failed_df)
else:
    print("\nAll calibration files processed successfully.")

display(
    calibration_summary_df[
        [
            "calibration_file",
            "calibration_n_annotations",
            "calibration_mean_north_deg",
            "calibration_circular_sd_deg",
            "true_north_video_deg",
            "calibration_quality_flag"
        ]
    ].sort_values("calibration_file")
)

Found 30 calibration files.

Saved robust raw calibration annotations to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_compass_calibrations\calibration_raw_annotations_ROBUST.csv

Saved robust calibration summary to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_compass_calibrations\calibration_summary_ROBUST.csv

All calibration files processed successfully.


,calibration_file,calibration_n_annotations,calibration_mean_north_deg,calibration_circular_sd_deg,true_north_video_deg,calibration_quality_flag
0,19_00022_waggle_annotations.csv,10,197.536749,1.093761e+00,218.806749,ok
1,19_00023_waggle_annotations.csv,10,166.058448,1.596858e+00,187.328448,ok
2,19_00047_waggle_annotations.csv,2,191.774435,3.705494e+00,213.044435,few_annotations
3,20_00014_waggle_annotations.csv,10,199.728881,1.307748e+00,220.998881,ok
4,20_00039_waggle_annotations.csv,1,182.556150,8.537736e-07,203.826150,only_one_annotation
5,20_00051_waggle_annotations.csv,10,190.913522,7.718931e-01,212.183522,ok
6,21_00020_waggle_annotations.csv,10,188.563043,1.421918e+00,209.833043,ok
7,21_00037_waggle_annotations.csv,10,199.758189,9.811943e-01,221.028189,ok
8,21_00040_waggle_annotations.csv,10,202.734980,1.029840e+00,224.004980,ok
9,22_00037_waggle_annotations.csv,10,183.323381,1.582208e+00,204.593381,ok
